# S1 — IS diagnosis (sealed OOS off-limits)

Forensics on the **frozen** STAR using **research IS only** (`is_research_is`).

Look for tails, cost drag, name concentration, early/late decay, and timing asserts **before** opening `09_oos_tearsheet.ipynb`. Disposition: kill / paper / new H. Do not re-tune on sealed OOS.


## 0. Imports & Config


In [ ]:
import os
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from dataclasses import replace

from backtest.s1_equities.costs import AlpacaCostModel, DEFAULT_COSTS, cost_summary
from backtest.s1_equities.portfolio import WEIGHT_EQUAL
from backtest.s1_equities.report import (
    concentration_vs_returns,
    metrics_table,
    plot_concentration,
    plot_cost_drag_vs_turnover,
    plot_drawdown,
    plot_equity_overlay,
    regime_performance,
    spy_weekly_open_returns,
    turnover_drag_summary,
)
from backtest.s1_equities.research import (
    DEFAULT_STAR_STACK,
    load_star_stack,
    n_trials_ledger_total,
    vt_target_ann_vol_from_stack,
)
from backtest.s1_equities.runner import run_backtest, summarize_periods
from backtest.s1_equities.signals import (
    default_backtest_paths,
    is_mask_from_preds,
    load_ohlc_panels,
    load_predictions,
    prediction_path_for_timing,
    score_matrix,
)
from backtest.s1_equities.stops import parse_stop_star
from backtest.s1_equities.timing import build_hold_table, exit_date_for_entry
from performance.sharpe_inference import return_moments
from risk.analytics.s1_equities.vol_targeting import parse_vol_target_star

stack = load_star_stack(DEFAULT_STAR_STACK)
print("BANNER: IS-only diagnosis — sealed OOS is off-limits")
print("STAR", {k: stack[k] for k in stack if k.endswith("_STAR")})
print("variant ledger arms", n_trials_ledger_total())
print("Disposition: kill / paper / new H — never re-tune on sealed OOS")


## 1. Contract

S1 trade-date panel: features known by close of `feature_date`; decision pre-open of trade `date`; fill at open of `date`. Costs are Alpaca-style (half-spread + open/close slippage). Periods are weekly.


In [ ]:
print("timing", stack["TIMING_STAR"])
print("N", stack["N_STAR"], "weight", stack["WEIGHT_STAR"], "stop", stack["STOP_STAR"])
print("VT", stack["VT_STAR"], "target_ann_vol", vt_target_ann_vol_from_stack(stack))
print("max_gross", stack.get("MAX_GROSS_STAR", 1.5))
display(pd.Series(cost_summary(DEFAULT_COSTS), name="alpaca_bps").to_frame("bps"))


## 2. Run frozen STAR on research IS only


In [ ]:
paths = default_backtest_paths(ROOT)
timing = str(stack["TIMING_STAR"])
pred_path = prediction_path_for_timing(paths, timing)
if not os.path.isfile(pred_path):
    raise FileNotFoundError(
        f"IS predictions missing: {pred_path}. Build them before this diagnosis."
    )
preds = load_predictions(pred_path)
is_mask = is_mask_from_preds(preds)
assert bool(is_mask.any()), "research IS mask is empty"
scores = score_matrix(preds)
opens, highs, lows, closes = load_ohlc_panels(paths["features"], tickers=list(scores.columns))
vt = replace(
    parse_vol_target_star(str(stack["VT_STAR"])),
    target_ann_vol=float(vt_target_ann_vol_from_stack(stack)),
)
stop = parse_stop_star(str(stack["STOP_STAR"]))
common_kw = dict(
    scores=scores,
    opens=opens,
    closes=closes,
    n=int(stack["N_STAR"]),
    timing_mode=timing,
    weight_mode=str(stack["WEIGHT_STAR"]),
    inv_vol_window=int(stack.get("INV_VOL_WINDOW_STAR", 42)),
    highs=highs,
    lows=lows,
    stop=stop,
    date_mask=is_mask,
    max_gross=float(stack.get("MAX_GROSS_STAR", 1.5)),
)
res = run_backtest(**common_kw, vol_target=vt, costs=DEFAULT_COSTS)
is_ret = res.period_returns.dropna().sort_index()
is_gross = res.period_returns_gross.reindex(is_ret.index).astype(float)
is_dates = pd.DatetimeIndex(is_ret.index)
print("IS periods", len(is_ret), "first", is_ret.index.min().date(), "last", is_ret.index.max().date())
print("stop_hit_rate", round(float(res.stop_hit_rate), 4), "mean_names_stopped", round(float(res.mean_names_stopped), 3))


## 3. Headline IS metrics (gross vs net) + tails


In [ ]:
net_tbl = metrics_table(is_ret, label="IS_net")
gross_tbl = metrics_table(is_gross, label="IS_gross")
display(pd.DataFrame([net_tbl, gross_tbl]).set_index("segment"))
mom = return_moments(is_ret, periods_per_year=52.0)
excess_kurt = float(mom["kurtosis"] - 3.0) if np.isfinite(mom["kurtosis"]) else float("nan")
tails = pd.Series(
    {
        "skew": mom["skew"],
        "excess_kurtosis": excess_kurt,
        "sr_period": mom["sr_period"],
        "n_obs": mom["n_obs"],
    },
    name="IS_tails",
)
display(tails.to_frame("value"))
if np.isfinite(excess_kurt) and excess_kurt > 5:
    warnings.warn(f"IS excess kurtosis {excess_kurt:.2f} — inspect tails before opening OOS")
if np.isfinite(net_tbl.get("max_drawdown", np.nan)) and float(net_tbl["max_drawdown"]) < -0.25:
    warnings.warn(f"IS max DD {float(net_tbl['max_drawdown']):.1%} breaches a 25% hole")

eq_net = res.equity.reindex(is_ret.index).dropna()
eq_gross = res.equity_gross.reindex(is_ret.index).dropna()
fig = plot_equity_overlay({"net": eq_net, "gross": eq_gross}, title="IS equity net vs gross")
plt.show()
fig = plot_drawdown(eq_net, title="IS drawdown (net)")
plt.show()


## 4. Early/late IS decay


In [ ]:
mid = len(is_ret) // 2
early = is_ret.iloc[:mid]
late = is_ret.iloc[mid:]
decay = pd.DataFrame(
    [
        metrics_table(early, label="early_IS"),
        metrics_table(late, label="late_IS"),
    ]
).set_index("segment")
display(decay)
early_s = float(decay.loc["early_IS", "sharpe"]) if "sharpe" in decay.columns else float("nan")
late_s = float(decay.loc["late_IS", "sharpe"]) if "sharpe" in decay.columns else float("nan")
ratio = late_s / early_s if np.isfinite(early_s) and abs(early_s) > 1e-9 else float("nan")
print("late/early Sharpe ratio", round(ratio, 3) if np.isfinite(ratio) else ratio)


## 5. Name concentration + attribution


In [ ]:
w_is = res.entry_weights.reindex(is_ret.index)
conc = concentration_vs_returns(w_is, is_ret)
display(conc["summary"].to_frame("value"))
display(conc["corr"])
fig = plot_concentration(conc["stats"], title="IS weight concentration")
plt.show()
# Top names by mean |weight| on IS
abs_mean = w_is.abs().mean().sort_values(ascending=False).head(15)
display(abs_mean.rename("mean_abs_w").to_frame())


## 6. Utilization + friction


In [ ]:
# turnover_drag_summary scores the complement of is_dates; build a dummy split
# so the "OOS" slice is the full research IS (all dates treated as after a null IS).
empty_is = pd.DatetimeIndex([])
drag = turnover_drag_summary(res, empty_is)
# Restrict frame to research IS entry dates only
frame = drag["frame"].reindex(is_ret.index).dropna(how="all")
to = res.turnover.reindex(is_ret.index).astype(float)
cost_drag = (is_gross - is_ret).rename("cost_drag")
summary = pd.Series(
    {
        "mean_turnover": float(to.mean()),
        "median_turnover": float(to.median()),
        "mean_cost_drag": float(cost_drag.mean()),
        "sharpe_net": float(summarize_periods(is_ret)["sharpe"]),
        "sharpe_gross": float(summarize_periods(is_gross)["sharpe"]),
        "stop_hit_rate": float(res.stop_hit_rate),
        "mean_names_stopped": float(res.mean_names_stopped),
        "mean_early_exit_notional": float(res.mean_early_exit_notional),
        "mean_gross_exposure": float(res.gross_exposure.reindex(is_ret.index).mean()),
    }
)
display(summary.to_frame("value"))
fig = plot_cost_drag_vs_turnover(
    pd.DataFrame({"turnover": to, "cost_drag": cost_drag, "ret_net": is_ret, "ret_gross": is_gross})
)
plt.show()


## 7. Cost stress (IS)


In [ ]:
rows = []
for mult in (1.0, 1.5, 2.0):
    stressed = AlpacaCostModel(
        commission_bps=DEFAULT_COSTS.commission_bps * mult,
        regulatory_bps=DEFAULT_COSTS.regulatory_bps * mult,
        half_spread_bps=DEFAULT_COSTS.half_spread_bps * mult,
        open_slippage_bps=DEFAULT_COSTS.open_slippage_bps * mult,
        close_slippage_bps=DEFAULT_COSTS.close_slippage_bps * mult,
        short_borrow_bps=DEFAULT_COSTS.short_borrow_bps * mult,
    )
    r = run_backtest(**common_kw, vol_target=vt, costs=stressed)
    sm = summarize_periods(r.period_returns)
    rows.append(
        {
            "cost_multiplier": mult,
            "ann_sharpe_net": sm["sharpe"],
            "max_drawdown": sm["max_drawdown"],
            "cagr": sm["cagr"],
            "open_one_way_bps": stressed.open_one_way_bps,
        }
    )
display(pd.DataFrame(rows))


## 8. Regimes (IS)


In [ ]:
spy_w = spy_weekly_open_returns(is_ret.index)
# Vol cutpoints fit on first half of IS; report performance on full IS.
cut_dates = is_dates[: max(3, len(is_dates) // 2)]
regimes = regime_performance(is_ret, spy_w, cut_dates)
print("Vol cutpoints (early IS):")
display(regimes["vol_cutpoints"].to_frame("value"))
print("By market direction:")
display(regimes["market"])
print("By vol tercile:")
display(regimes["vol"])


## 9. Baselines

S1-native only: equal-weight book without VT, and STAR weights without stops. Not trad-z / TSMOM / G10.


In [ ]:
base_rows = []
# Frozen STAR (already run)
base_rows.append({"arm": "frozen_star", **summarize_periods(is_ret)})
# Equal weight, same N/timing/stop/VT
eq = run_backtest(**{**common_kw, "weight_mode": WEIGHT_EQUAL}, vol_target=vt, costs=DEFAULT_COSTS)
base_rows.append({"arm": "equal_weight", **summarize_periods(eq.period_returns)})
# No vol target
no_vt = run_backtest(**common_kw, vol_target=None, costs=DEFAULT_COSTS)
base_rows.append({"arm": "no_vt", **summarize_periods(no_vt.period_returns)})
# No stops
no_stop = run_backtest(**{**common_kw, "stop": None}, vol_target=vt, costs=DEFAULT_COSTS)
base_rows.append({"arm": "no_stop", **summarize_periods(no_stop.period_returns)})
display(pd.DataFrame(base_rows).set_index("arm")[["sharpe", "max_drawdown", "cagr", "vol"]])


## 10. Timing / hold assert


In [ ]:
cal = pd.DatetimeIndex(pd.to_datetime(opens.index)).sort_values().unique()
holds = res.hold_table.copy()
if holds.empty:
    holds = build_hold_table(res.entry_weights, timing, trading_calendar=cal)
holds = holds.copy()
holds["entry_date"] = pd.to_datetime(holds["entry_date"])
holds["exit_date"] = pd.to_datetime(holds["exit_date"])
holds = holds.loc[holds["entry_date"].isin(is_dates)].copy()
holds["ok_exit_after_entry"] = holds["exit_date"] > holds["entry_date"]
holds["ok_entry_weekday"] = holds["entry_date"].dt.weekday == 0  # Monday
holds["ok_exit_recomputed"] = [
    pd.Timestamp(exit_date_for_entry(
        ed, timing, entry_dates=res.entry_weights.index, trading_calendar=cal
    )) == pd.Timestamp(xd)
    for ed, xd in zip(holds["entry_date"], holds["exit_date"])
]
checks = {
    "n_holds": len(holds),
    "pct_exit_after_entry": float(holds["ok_exit_after_entry"].mean()) if len(holds) else float("nan"),
    "pct_monday_entry": float(holds["ok_entry_weekday"].mean()) if len(holds) else float("nan"),
    "pct_exit_matches_timing": float(holds["ok_exit_recomputed"].mean()) if len(holds) else float("nan"),
}
display(pd.Series(checks, name="timing_assert").to_frame("value"))
display(holds.head())
if checks["n_holds"] and checks["pct_exit_after_entry"] < 1.0:
    warnings.warn("Some holds have exit_date <= entry_date")
if checks["n_holds"] and checks["pct_exit_matches_timing"] < 1.0:
    warnings.warn("Some hold exits disagree with timing.exit_date_for_entry")


## 11. Cause-class checklist + post-mortem stub

**Cause class (pick one):** concentration / tails / costs / VT overfit / IC decay / regime / bug / no edge

**Disposition:** kill / paper / new H-id (do not re-test sealed OOS after recipe changes)


In [ ]:
print("Paste into the hyp log / notes after review:")
print(
    f"""
### Post-mortem — S1 STAR IS diagnosis

- Date:
- IS net Sharpe: {float(summarize_periods(is_ret)['sharpe']):.3f}
- IS gross Sharpe: {float(summarize_periods(is_gross)['sharpe']):.3f}
- Excess kurtosis: {excess_kurt:.2f}
- Max |w| mean: {float(conc['summary'].get('max_abs_w_mean', float('nan'))):.4f}
- Mean turnover: {float(to.mean()):.4f}
- Cost stress 2.0x Sharpe: (see §7)
- Late/early Sharpe ratio: {ratio if np.isfinite(ratio) else 'nan'}

**Cause class:** concentration / tails / costs / VT overfit / IC decay / regime / bug / no edge

**Evidence:**
-

**Disposition:** kill / paper / new H-id
"""
)
